# 08   VORIGE VERSIE FE
~Feature Engineering~


___

**Input:**  `data_processed/train.parquet` · `data_processed/holdout.parquet`  
**Output:** `data_features/train_features.parquet`  
           `data_features/holdout_features.parquet`  
           `data_features/feature_schema.json`  
           `data_features/transformers/`

**info**
[[08d_feature_engineering_tabel.md]] <-- briefing &  tabel
Doel, TSEL-framework, immutability-contract, input/output contract

Tabel: welke transformers fitten op train → transformers/ map


**Structuurprincipes**

- **Fit uitsluitend op `train`**, transform op beide — elke sectie vermeldt dit expliciet
- **Chronologische volgorde**: T → S → E → L (lags als laatste, want time-aware)
- **Freeze-point**: na cel 10 verandert er niets meer aan features in enig later notebook

## Architectuur: T · S · E · L

| Blok | Inhoud | Fitting op train? |
|------|--------|-------------------|
| **T** — Time structure | sin/cos cyclisch, kalender, day_type_3 | ❌ |
| **S** — Spatial identity | tier, cluster, log_capacity, one-hot parking | ❌ |
| **E** — External | weer (binning/scaling), events, interacties, cascade | ✅ scaler + bins |
| **L** — Lags | occ_lag_1h / 24h / 168h (time-aware) | ❌ |

> ⚠ **FREEZE POINT:** na cel 10 worden features niet meer aangepast in enig volgend notebook.  
> Transformatieparameters worden **uitsluitend geschat op `train`** en daarna toegepast op `holdout`.

___
## Wat is feature engineering?

Feature engineering is het proces waarbij ruwe data wordt omgezet in informatieve inputvariabelen (*features*) die een machine learning-model in staat stellen relevante patronen te leren. Het gaat niet om het verzamelen van meer data, maar om het intelligenter representeren van bestaande data. In tijdreekscontexten — zoals parkeerpredicties — is het de meest bepalende stap voor modelkwaliteit, vaak belangrijker dan de keuze van het model zelf (Cerqueira et al., 2023). [sciencedirect](https://www.sciencedirect.com/science/article/pii/S2352146523012206)

**Academische formulering voor de thesis:**

> *"Feature engineering transformeert ruwe observaties in gestructureerde representaties die de onderliggende data-genererende processen blootleggen voor het leeralgoritme. In tijdreeksapplicaties omvat dit doorgaans vier categorieën: temporele structuurfeatures, lag-features, externe covariaten en ruimtelijke identiteitsfeatures (Wan et al., 2023; Cerqueira et al., 2023)."*

***

## De vier categorieën van features in nb08

### Categorie 1 — Temporele structuurfeatures (~8 features)

Dit zijn features die de **cyclische en kalenderstructuur** van de tijdreeks representeren. Parking-bezetting vertoont sterke dagelijkse en wekelijkse periodiciteit (H-T1, H-T2 bevestigd) en is daardoor bij uitstek geschikt voor cyclische encoding.

**Waarom sin/cos en niet gewoon het uurcijfer?**
Een ruwe uurfeature (`hour = 23`) en (`hour = 0`) liggen numeriek ver uiteen, terwijl ze gedragsmatig naast elkaar liggen (middernacht). Cyclische sin/cos-encoding lost dit op door de cirkelstructuur te respecteren (Wan et al., 2023): [mavmatrix.uta](https://mavmatrix.uta.edu/cgi/viewcontent.cgi?article=1010&context=utalibraries_openinitiativespubs)

$\text{hour\_sin} = \sin\!\left(\frac{2\pi \cdot h}{24}\right), \quad \text{hour\_cos} = \cos\!\left(\frac{2\pi \cdot h}{24}\right)$

| Feature | Periode | Literatuurbasis |
|---|---|---|
| `hour_sin`, `hour_cos` | 24u cyclus | Wan et al. (2023); Cerqueira et al. (2023) |
| `weekday_sin`, `weekday_cos` | 7-daagse cyclus | Fokker et al. (2021) |
| `month_sin`, `month_cos` | 12-maandse cyclus | H-T1 seizoenspatroon |
| `day_type_3` | Weekdag / zaterdag / zondag-feestdag | H-T1; Zhang et al. (2020) |
| `year_dummy` | Categorisch 2020=0, 2023/24=1 | H-T4 COVID-shift |

**Statistiek die nodig is:** geen fitting — louter wiskundige transformatie. Géén risk op leakage.

***

### Categorie 2 — Lag-features / autoregressive features (~3 features)

Dit zijn de **sterkste predictoren** in het model. Ze representeren de bezetting op eerdere tijdsmomenten en exploiteren de autocorrelatie die in H-T2 is aangetoond. De literatuur is eenduidig: historische bezetting is de meest informatieve predictor voor toekomstige bezetting (Lira et al., 2021; Yang et al., 2019). [ietresearch.onlinelibrary.wiley](https://ietresearch.onlinelibrary.wiley.com/doi/full/10.1049/itr2.12433)

| Feature | Lag | Motivatie |
|---|---|---|
| `occ_lag_1h` | t − 1 uur | Sterke positieve ACF bij lag-1 |
| `occ_lag_24h` | t − 24 uur | Dagelijkse periodiciteit (H-T2) |
| `occ_lag_168h` | t − 168 uur | Wekelijkse periodiciteit (H-T2) |

**Statistiek:** geen fitting — puur temporele verschuiving via `DataFrame.shift()`. Wel: expliciete NaN-check voor de eerste 168 uur van de holdout.

**⚠ Leakage-risico:** lag-features moeten worden aangemaakt *na* de train/holdout-split, op elk deel afzonderlijk. Dit is al geregeld via `train.parquet` en `holdout.parquet` uit nb07c.

***

### Categorie 3 — Externe covariaten (~10–12 features)

Dit zijn features die **externe invloeden** representeren: weer, kalender, events. Ze voegen signaal toe bovenop de structurele tijdreekspatronen.

**Weersfeatures:**

| Feature | Encoding | Motivatie | Statistiek nodig |
|---|---|---|---|
| `precip_bin` | 4 bins (droog/licht/matig/zwaar) | H-E1: drempelpatroon | Binning op trainingsdistributie |
| `wind_dummy` | 0/1 (>10 m/s) | H-E6: modal shift | Drempel hard-coded |
| `sun_duration_min` | Gestandaardiseerd | H-E7: laag prioriteit | Fit StandardScaler op train |
| ~~`temp_c`~~ | ~~Verwijderd~~ | Globale ρ = −0.013, Simpson's Paradox | — |

**Kalenderfeatures:**

| Feature | Encoding | Motivatie |
|---|---|---|
| `is_school_vacation` | Binair | H-E8: sterkste kalendereffect |
| `tier × is_school_vacation` | Interactieproduct | H-E8: Δr_rb = +0.117 |
| `is_national_holiday × tier` | Interactieproduct | H-E4: hertoetsen na Fischer z-fix |

**Event-features** (afgeleid via keyword-matching op `event_names_combined`):

| Feature | Encoding | Prioriteit |
|---|---|---|
| `is_kermis` | Binair | 🔴 Hoog — stadswijd medium effect |
| `is_voetbal × tier` | Interactieproduct | 🔴 Hoog — substitutie-effect |
| `is_festival × tier` | Interactieproduct | 🔴 Hoog — sterkste centrum-negatief |
| `is_carnaval × tier` | Interactieproduct | 🟡 Medium |
| `is_processie` | Binair | 🟡 Medium |
| `hours_to_next_event` | Continu (uren) | 🟡 Medium — cascade H-E3 |
| `hours_since_last_event` | Continu (uren) | 🟡 Medium — cascade H-E3 |

**Statistiek:** `StandardScaler` voor continue weersfeatures — **fit uitsluitend op trainingsset**, transform op beide.

***

### Categorie 4 — Ruimtelijke identiteitsfeatures (~4–12 features)

Dit zijn features die **welke parking** het is representeren. Ze zijn de dominantste ruimtelijke predictor (H-S1: η² klein maar RF-importance hoog).

| Feature | Encoding | Motivatie | Statistiek nodig |
|---|---|---|---|
| `parking_id` | Target encoding (regularized) | Dominante ruimtelijke predictor | Fit op train only — leakage-risico! |
| `tier_admin` | One-hot (2) | Administratieve tierindeling | Geen |
| `cluster_data` | Binair (0/1) | Data-gedreven k=2, silhouette=0.615 | Geen |
| `high_lt_pressure` | Binair | Ceiling effect P Hoogstraat/P Komet | Geen |

**Waarom target encoding voor `parking_id` en niet one-hot?**
Met 10 parkings is one-hot (10 kolommen) technisch haalbaar, maar target encoding is in de literatuur empirisch superieur voor lage-kardinaliteit categorische features in boomgebaseerde modellen (Pargent et al., 2021): het comprimeert de parkeerspelling naar één informatiedichte kolom en reduceert het risico op overfitting.  **Kritisch:** target encoding moet worden berekend met leave-one-out of k-fold cross-validation op de trainingsset om leakage te vermijden — scikit-learn's `TargetEncoder` doet dit automatisch. [arxiv](https://arxiv.org/abs/2104.00629v2)

***

## Totaaltelling: hoeveel features?

| Categorie | Aantal | Prioriteit |
|---|---|---|
| Temporele structuurfeatures | 8 | Allemaal 🔴 Hoog |
| Lag-features | 3 | Allemaal 🔴 Hoog |
| Externe covariaten (weer + kalender + events) | 12–14 | 🔴 Hoog tot 🟡 Medium |
| Ruimtelijke identiteitsfeatures | 4 | 🔴 Hoog |
| **Totaal ingaande feature-matrix** | **~27–29** | |
| Na VIF-pruning en importance-filter (verwacht) | **~22–25** | |

***

## Statistische stappen in nb08 — in volgorde

1. **Keyword-classificatie** `event_label_derived` op `event_names_combined` — geen statistiek, louter string-matching
2. **Lag-constructie** via `shift()` per `parking_id` — geen statistiek, NaN-check achteraf
3. **Cyclische transformatie** `sin/cos` — wiskundige formule, geen fitting
4. **Binning neerslag** `pd.cut()` met RMI-grenzen — fit op trainingsdistributie, apply op holdout
5. **StandardScaler** voor `wind_speed_ms`, `sun_duration_min` — fit op train, transform op beide
6. **Target encoding** `parking_id` — fit op train via k-fold, transform op beide (Pargent et al., 2021) [arxiv](https://arxiv.org/abs/2104.00629v2)
7. **Interactietermen** als rekenkundig product — geen statistiek
8. **VIF-herberekening** op volledige feature-matrix inclusief lag-features en interacties — multicollineariteitscheck
9. **Exporteren** als `train_features.parquet` en `holdout_features.parquet`

***

## Literatuursamenvatting voor de thesis-methodologie

> *"De feature engineering pipeline volgt vier categorieën die in de parking-predictieliteratuur empirisch zijn gevalideerd: (1) cyclische temporele encodings via sin/cos-transformatie voor dag-, week- en jaarcycli (Wan et al., 2023; Cerqueira et al., 2023), (2) autoregressive lag-features op tijdsafstanden van 1, 24 en 168 uur conform de empirisch vastgestelde ACF-structuur (Lira et al., 2021; Yang et al., 2019), (3) externe covariaten voor weer, kalender en event-cascade-effecten (Fokker et al., 2021; Oz, 2023), en (4) ruimtelijke identiteitsfeatures via geregulariseerde target encoding van parking_id (Pargent et al., 2021). Alle transformatieparameters worden uitsluitend geschat op de trainingsset (2020, 2023, 2024) en vervolgens toegepast op de holdoutset (2025) om temporele dataleakage te voorkomen (Cerqueira et al., 2023)."* [ietresearch.onlinelibrary.wiley](https://ietresearch.onlinelibrary.wiley.com/doi/full/10.1049/itr2.12433)

## De drie echte opties

**Optie A — Één model + tier als feature** *(huidige aanpak)*
- Voordelen: alle data samen, model leert tier-effecten via interacties, SHAP ontleedt achteraf
- Nadeel: minder interpreteerbaar voor beleidsmakers die "centrum vs. vesten" willen vergelijken

**Optie B — Aparte modellen op administratieve tier**
- Voordelen: eenvoudig, interpreteerbaar, voldoende data per model
- Nadeel: drie afwijkende parkings (Keerdok, Tinel, Hoogstraat) vervuilen beide modellen systematisch

**Optie C — Aparte modellen op data-gedreven cluster** *(methodologisch sterkst)*
- Cluster 0: P Grote Markt, P Kathedraal, P Lamot, P Veemarkt, P Keerdok, P Tinel → **6 parkings, ~78.000 rijen**
- Cluster 1: P Bruul, P Komet, P Maarten, P Hoogstraat → **4 parkings, ~52.000 rijen**
- Voordelen: gedragsmatig coherente groepen, silhouette = 0.615 valideert de split
- Nadeel: minder intuïtief voor beleidsmakers ("cluster 0" zegt niets zonder toelichting)

***

## Aanbevolen

**Optie A als hoofdmodel + Optie B als robustheidscheck.**

Concreet:
- Nb09–nb11: train één model met tier als feature (Optie A) — dit is je primaire resultaat
- Nb12 (SHAP): analyseer SHAP-waarden per administratieve tier — dit geeft de beleidsrelevante interpretatie
- Nb12 addendum: train ook twee aparte modellen per administratieve tier (Optie B) en vergelijk de MAPE — als de prestaties vergelijkbaar zijn, valideer je daarmee de tierstructuur; als Optie A beter scoort, heb je empirisch bewijs dat één geïntegreerd model de tierinteracties beter leert

**Academische formulering:**

> *"Hoewel een stratificatie naar administratieve tier (centrum vs. vesten/rand) beide deelmodellen voldoende data zou bieden (~65.000 respectievelijk ~57.000 trainingsrijen), toont de data-gedreven clusteranalyse (nb06) dat drie parkings gedragsmatig afwijken van hun administratieve label. Een gestratificeerde aanpak op administratieve basis introduceert bijgevolg systematische misclassificatie voor P Keerdok, P Tinel en P Hoogstraat. Als primaire modelarchitectuur wordt gekozen voor één geïntegreerd model met tier als expliciete feature en interactietermen; een gestratificeerd model per administratieve tier wordt als robustness check gerapporteerd in nb12."*



____

In [1]:
# ── Cel 01 · Imports & data laden ─────────────────────────────────────
from pathlib import Path
import pandas as pd
import numpy as np

ROOT      = Path("/Users/emilevandevoorde/Documents/mechelen_parking")
DATA_PROC = ROOT / "data_processed"

train   = pd.read_parquet(DATA_PROC / "train.parquet")
holdout = pd.read_parquet(DATA_PROC / "holdout.parquet")

# tier_admin aanmaken (conform nb07c)
for df in [train, holdout]:
    df["tier_admin"] = (
        df["parking_location_category"]
        .astype(str)
        .replace({"rand": "vesten_of_rand", "vesten": "vesten_of_rand"})
    )

print(f"Train  : {len(train):>7,} rijen × {train.shape[1]} kolommen")
print(f"Holdout: {len(holdout):>7,} rijen × {holdout.shape[1]} kolommen")


Train  : 129,837 rijen × 64 kolommen
Holdout:  87,600 rijen × 64 kolommen


## dtype audit

In [2]:
# ── Dtype audit: train vs. holdout ───────────────────────────────────
import pandas as pd

def dtype_audit(df, name):
    summary = pd.DataFrame({
        "dtype":    df.dtypes,
        "n_null":   df.isnull().sum(),
        "pct_null": (df.isnull().sum() / len(df) * 100).round(2),
        "n_unique": df.nunique(),
        "sample":   [df[c].dropna().iloc[0] if df[c].notna().any() 
                     else "ALL NULL" for c in df.columns]
    })
    print(f"\n{'='*60}")
    print(f"  {name}  —  {df.shape[0]:,} rijen × {df.shape[1]} kolommen")
    print(f"{'='*60}")
    print(summary.to_string())
    return summary

audit_train   = dtype_audit(train,   "TRAIN   (2020+2023+2024)")
audit_holdout = dtype_audit(holdout, "HOLDOUT (2025)")

# ── Vergelijk dtypes train vs. holdout ───────────────────────────────
print(f"\n{'='*60}")
print("  DTYPE MISMATCH train vs. holdout")
print(f"{'='*60}")
mismatch = (
    pd.DataFrame({
        "train_dtype":   audit_train["dtype"],
        "holdout_dtype": audit_holdout["dtype"]
    })
    .query("train_dtype != holdout_dtype")
)
if mismatch.empty:
    print("✅ Geen dtype-mismatches gevonden")
else:
    print(mismatch.to_string())

# ── Kolommen enkel in train of enkel in holdout ───────────────────────
only_train   = set(train.columns)   - set(holdout.columns)
only_holdout = set(holdout.columns) - set(train.columns)

print(f"\n{'='*60}")
print("  KOLOMMEN ENKEL IN TRAIN:")
print(only_train if only_train else "  ✅ Geen")
print("\n  KOLOMMEN ENKEL IN HOLDOUT:")
print(only_holdout if only_holdout else "  ✅ Geen")

# ── Specifieke checks voor nb08-relevante kolommen ────────────────────
print(f"\n{'='*60}")
print("  NB08-RELEVANTE KOLOMMEN — detail")
print(f"{'='*60}")
nb08_cols = [
    "rounded_hour", "occupancy_rate", "parking_id", "tier_admin",
    "parking_location_category", "year", "month", "hour", "weekday",
    "day_type_3", "temp_c", "precip_mm", "wind_speed_ms",
    "sun_duration_min", "is_event_day", "event_names_combined",
    "event_scale_max", "n_concurrent_events", "is_school_vacation",
    "is_national_holiday", "low_data_coverage", "system_blackout"
]
for col in nb08_cols:
    if col in train.columns:
        t_dtype  = train[col].dtype
        t_null   = train[col].isnull().sum()
        t_sample = train[col].dropna().iloc[0] if train[col].notna().any() else "NULL"
        print(f"  {col:<30} {str(t_dtype):<15} nulls={t_null:<6} sample={t_sample}")
    else:
        print(f"  {col:<30} ⚠ NIET AANWEZIG in train")



  TRAIN   (2020+2023+2024)  —  129,837 rijen × 64 kolommen
                                    dtype  n_null  pct_null  n_unique                                                            sample
parking_id                            str       0      0.00        10                                                           P Bruul
parking_id_hash                  category       0      0.00        10  0cdac8cfff7924a40f8ec805a661afd5663ee08c7ff707353430c982eb4b6d7b
parking_type                          str       0      0.00         1                                                       Car Parking
kind                                  str       0      0.00         1                                                         shortterm
year                                int32       0      0.00         3                                                              2020
month                               int32       0      0.00        12                                                       

# Notebook 08 — Feature Engineering Pipeline

**Project:** Tier-Stratified Occupancy Prediction and Scenario-Based Policy Simulation  
in Urban Parking Systems — A Case Study of Mechelen  
**Auteur:** Emile Van de Voorde  
**Input:** `data_processed/train.parquet` · `data_processed/holdout.parquet`  
**Output:** `data_processed/train_features.parquet` · `data_processed/holdout_features.parquet`

---

Deze notebook construeert de definitieve feature-matrix voor alle modelleer- en 
simulatienotebooks (nb09–nb13). Alle transformatieparameters worden uitsluitend 
geschat op de trainingsset (2020, 2023, 2024) en vervolgens toegepast op de 
holdoutset (2025) om temporele dataleakage te voorkomen (Cerqueira et al., 2023).

De feature-keuzes zijn empirisch onderbouwd door de 17 hypothesentoetsen uit 
FASE02 (nb05–nb07b) en bijgesteld via vier post-hoc analyses. Zie nb08-header 
markdown voor de volledige motivatie per feature.


### Voorbereidend
- `day_type_3` afleiden uit `weekday_int` + `is_national_holiday` (kolom bestaat nog niet)
- `tier_num` aanmaken als binaire integer van `tier_admin` (centrum=1, vesten=0) — nodig voor interacties

### Cyclische temporele encodings
- `hour_sin` + `hour_cos` uit `hour` (24u cyclus)
- `weekday_sin` + `weekday_cos` uit `weekday_int` (7-daagse cyclus)
- `month_sin` + `month_cos` uit `month` (12-maandse cyclus)

### Lag-features ⚠ leakage-kritisch
- Eerst sorteren op `parking_id` + `rounded_hour`
- `occ_lag_1h`, `occ_lag_24h`, `occ_lag_168h` via `groupby("parking_id").shift()`
- NaN-check na aanmaak — eerste 168 rijen per parking zijn verwacht NaN, droppen in nb09

### Weersfeatures ⚠ fit op train only
- `precip_bin`: 4 categorieën op vaste RMI-grenzen (0 / 2 / 5 mm) — geen fitting
- `wind_strong`: binaire drempel >10 m/s — geen fitting
- `sun_duration_scaled`: StandardScaler — **fit op train, transform op beide**
- `temp_c` **niet opnemen** — globale ρ = −0.013, Simpson's Paradox

### Ruimtelijke features ⚠ fit op train only
- `cluster_data`: hard-coded map uit nb06-resultaat (cluster 0 vs. 1)
- `high_lt_pressure`: binair — P Hoogstraat en P Komet = 1, rest = 0
- `parking_id_encoded`: TargetEncoder (cv=5) — **fit op train, transform op beide**

### Event-features
- Event-dummies **bestaan al**: `is_football_day`, `is_festival_day`, `is_kermis_day`, `is_carnival_day`, `is_procession_day`
- `is_event_day` generieke dummy **niet** opnemen als standalone feature — maskeert tegengestelde tier-effecten

### Interactietermen (rekenkundig product — geen fitting)
- `voetbal_x_centrum` = `is_football_day` × `tier_num`
- `festival_x_centrum` = `is_festival_day` × `tier_num`
- `carnaval_x_centrum` = `is_carnival_day` × `tier_num`
- `vakantie_x_centrum` = `is_school_vacation` × `tier_num`
- `feestdag_x_centrum` = `is_national_holiday` × `tier_num`

### Cascade-features
- `hours_to_next_event` en `hours_since_last_event` afleiden uit `is_event_day` per parking — forward/backward iteratie per groep

### Kalender
- `year_2020`: binair (2020=1, rest=0) — categorisch, **niet** ordinaal
- `is_school_vacation` en `is_national_holiday` bestaan al als float — casten naar int

### Kwaliteitscontroles
- VIF-check op volledige feature-matrix — alles >10 onderzoeken
- Dtype-check: alle features moeten float of int zijn vóór export — geen strings, geen categoricals
- Kolom-symmetrie bevestigen: train en holdout bevatten exact dezelfde feature-kolommen

### Export
- `train_features.parquet` en `holdout_features.parquet` opslaan naar `data_processed/`
- `rounded_hour`, `parking_id`, `tier_admin`, `year` mee exporteren als metadata — niet als modelfeature

___

## Stap 1 — Voorbereidende kolomconstructie

### 1.1 `day_type_3`

De bestaande `day_type`-kolom maakt enkel onderscheid tussen `'Weekday'` en 
`'Weekend'`, wat onvoldoende is voor het parkeergedrag in Mechelen. Uit H-T1 
(FASE02) bleek dat zaterdag structureel verschilt van zondag en nationale 
feestdagen: zaterdagen vertonen een commercieel retailprofiel met een 
middagpiek, terwijl zondagen en feestdagen een gedrukt, vlak profiel tonen 
vergelijkbaar met vakantiedagen (Zhang et al., 2020; Fokker et al., 2021).

De drie klassen zijn:
- `weekday` — maandag t.e.m. vrijdag, geen nationale feestdag
- `saturday` — zaterdag, geen nationale feestdag
- `sunday_holiday` — zondag of nationale feestdag (beide vertonen identiek gedragsprofiel)

### 1.2 `tier_num`

De string-kolom `tier_admin` wordt omgezet naar een binaire integer die nodig 
is als factor in de interactietermen (Stap 6). Zonder numerieke representatie 
kan het product `is_football_day × tier` niet worden berekend.


In [3]:
# ── Cel 02 · Voorbereidende kolomconstructie ──────────────────────────

for df in [train, holdout]:

    # ── day_type_3 ────────────────────────────────────────────────────
    df["day_type_3"] = df["weekday_int"].map({
        0: "weekday", 1: "weekday", 2: "weekday",
        3: "weekday", 4: "weekday",
        5: "saturday",
        6: "sunday_holiday"
    })
    # Nationale feestdag overschrijft altijd → sunday_holiday
    df.loc[df["is_national_holiday"] == 1, "day_type_3"] = "sunday_holiday"

    # ── tier_num ──────────────────────────────────────────────────────
    df["tier_num"] = (df["tier_admin"] == "centrum").astype(int)

# ── Verificatie ───────────────────────────────────────────────────────
print("day_type_3 verdeling (train):")
print(train["day_type_3"].value_counts())
print()
print("tier_num verdeling (train):")
print(train.groupby(["parking_id", "tier_num", "tier_admin"])
      .size().reset_index(name="n")
      .to_string(index=False))


day_type_3 verdeling (train):
day_type_3
weekday           90385
sunday_holiday    21928
saturday          17524
Name: count, dtype: int64

tier_num verdeling (train):
   parking_id  tier_num     tier_admin     n
      P Bruul         0 vesten_of_rand 12709
P Grote Markt         1        centrum 19044
 P Hoogstraat         1        centrum 12954
 P Kathedraal         1        centrum 12232
    P Keerdok         0 vesten_of_rand 16810
      P Komet         0 vesten_of_rand  3126
      P Lamot         1        centrum 14013
    P Maarten         0 vesten_of_rand  5944
      P Tinel         0 vesten_of_rand 18155
   P Veemarkt         1        centrum 14850


## Stap 2 — Cyclische temporele encodings

Parkeergedrag vertoont sterke periodieke structuren op drie tijdsschalen: 
dagelijks (ochtend-/avondspits), wekelijks (weekdag vs. weekend) en jaarlijks 
(seizoenen, schoolvakanties). Ruwe numerieke representatie van deze cycli 
(bijv. hour=23 en hour=0) introduceert een artificiële discontinuïteit aan 
de grens van elke cyclus, terwijl de twee tijdsmomenten gedragsmatig naast 
elkaar liggen.

De standaardoplossing in de tijdreeksliteratuur is sinusoïdale encoding 
(Wan et al., 2023; Cerqueira et al., 2023):

$$\text{feature\_sin} = \sin\!\left(\frac{2\pi \cdot t}{T}\right), \quad
\text{feature\_cos} = \cos\!\left(\frac{2\pi \cdot t}{T}\right)$$

waarbij $T$ de periode is (24 voor uur, 7 voor weekdag, 12 voor maand). 
Het sin/cos-paar samen beschrijft een unieke positie op de cirkel — 
één component alleen is onvoldoende omdat bijv. uur=6 en uur=18 
dezelfde sinuswaarde geven maar gedragsmatig totaal verschillen.

> **Geen fitting vereist** — louter wiskundige transformatie, identiek 
> toegepast op train en holdout. Geen leakage-risico.


In [4]:
# ── Cel 03 · Cyclische temporele encodings ────────────────────────────

for df in [train, holdout]:
    # Uur (T = 24)
    df["hour_sin"]    = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]    = np.cos(2 * np.pi * df["hour"] / 24)
    # Weekdag (T = 7)
    df["weekday_sin"] = np.sin(2 * np.pi * df["weekday_int"] / 7)
    df["weekday_cos"] = np.cos(2 * np.pi * df["weekday_int"] / 7)
    # Maand (T = 12)
    df["month_sin"]   = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"]   = np.cos(2 * np.pi * df["month"] / 12)

# ── Verificatie: range moet [-1, 1] zijn, geen nulls ─────────────────
cyc_cols = ["hour_sin","hour_cos","weekday_sin",
            "weekday_cos","month_sin","month_cos"]

check = pd.DataFrame({
    "min":    [train[c].min()    for c in cyc_cols],
    "max":    [train[c].max()    for c in cyc_cols],
    "nulls":  [train[c].isnull().sum() for c in cyc_cols],
}, index=cyc_cols).round(6)

print("Cyclische encodings — range & null check (train):")
print(check.to_string())


Cyclische encodings — range & null check (train):
                  min       max  nulls
hour_sin    -1.000000  1.000000      0
hour_cos    -1.000000  1.000000      0
weekday_sin -0.974928  0.974928      0
weekday_cos -0.900969  1.000000      0
month_sin   -1.000000  1.000000      0
month_cos   -1.000000  1.000000      0


## Stap 3 — Lag-features (autoregressive)

Autoregressive lag-features zijn empirisch de sterkste predictoren in 
parkeer-occupancy modellen (Lira et al., 2021; Yang et al., 2019). 
De ACF-analyse in H-T2 (nb07) bevestigde drie significante pieken:

| Lag | Tijdsafstand | Gedragsmatige motivatie |
|-----|-------------|------------------------|
| 1   | 1 uur       | Sterke kortetermijn autocorrelatie |
| 24  | 24 uur      | Dagelijkse periodiciteit |
| 168 | 1 week      | Wekelijkse periodiciteit |

**Kritische vereisten:**
- Sorteren op `parking_id` + `rounded_hour` vóór shift — anders worden 
  waarden van parking A gebruikt als lag voor parking B
- Aanmaken ná de train/holdout-split — anders lekt holdout-informatie 
  achterwaarts in de trainingsset
- NaN bij de eerste 1/24/168 rijen per parking zijn **correct en verwacht** 
  — deze worden in nb09 gedropped vóór modeltraining


In [5]:
# ── Cel 04 · Lag-features ─────────────────────────────────────────────

# Sorteren is verplicht voor correcte shift per parking
train   = train.sort_values(
    ["parking_id", "rounded_hour"]).reset_index(drop=True)
holdout = holdout.sort_values(
    ["parking_id", "rounded_hour"]).reset_index(drop=True)

for df in [train, holdout]:
    grp = df.groupby("parking_id")["occupancy_rate"]
    df["occ_lag_1h"]   = grp.shift(1)
    df["occ_lag_24h"]  = grp.shift(24)
    df["occ_lag_168h"] = grp.shift(168)

# ── NaN-check ─────────────────────────────────────────────────────────
print("NaN-check lag-features:\n")
print(f"{'Feature':<16} {'Train nulls':>12} {'Holdout nulls':>14} "
      f"{'Verwacht (train)':>18}")
for lag, expected in [("occ_lag_1h",10),("occ_lag_24h",10),("occ_lag_168h",10)]:
    nt = train[lag].isnull().sum()
    nh = holdout[lag].isnull().sum()
    print(f"{lag:<16} {nt:>12} {nh:>14}      {expected} × 1 per parking")

print()
# Controleer dat nulls zich beperken tot de eerste rijen per parking
print("NaN-rijen occ_lag_168h in train — uitsluitend eerste rijen per parking?")
null_check = (
    train[train["occ_lag_168h"].isnull()]
    .groupby("parking_id")
    .agg(n_nulls=("rounded_hour","count"),
         eerste=("rounded_hour","min"),
         laatste=("rounded_hour","max"))
)
print(null_check.to_string())


NaN-check lag-features:

Feature           Train nulls  Holdout nulls   Verwacht (train)
occ_lag_1h                 10             10      10 × 1 per parking
occ_lag_24h               240            240      10 × 1 per parking
occ_lag_168h             1680           1680      10 × 1 per parking

NaN-rijen occ_lag_168h in train — uitsluitend eerste rijen per parking?
               n_nulls              eerste             laatste
parking_id                                                    
P Bruul            168 2020-01-01 00:00:00 2020-01-08 00:00:00
P Grote Markt      168 2020-01-01 00:00:00 2020-01-09 05:00:00
P Hoogstraat       168 2023-01-01 00:00:00 2023-01-11 07:00:00
P Kathedraal       168 2020-01-01 21:00:00 2020-08-23 03:00:00
P Keerdok          168 2023-01-17 00:00:00 2023-01-24 06:00:00
P Komet            168 2024-08-23 01:00:00 2024-08-30 17:00:00
P Lamot            168 2020-01-01 00:00:00 2020-01-09 10:00:00
P Maarten          168 2024-04-26 16:00:00 2024-05-03 15:00:00
P

## Stap 3b — Diagnose tijdreeksgaten in lag-features

Vóór we lag-features gebruiken in het model, moeten we begrijpen 
waar de NaN-waarden vandaan komen. Een `shift(168)` over een 
tijdreeksgat (bijv. parking start pas in augustus 2024) produceert 
een NaN die *niet* de eerste week van de reeks vertegenwoordigt, 
maar een structurele onderbreking. Zulke NaN-rijen mogen niet 
worden geïmputeerd — ze moeten worden gedropped in nb09, net zoals 
de initiële opstartNaN's.

Het onderscheid is methodologisch belangrijk: als we naïef imputeren,
injecteren we fictieve autocorrelatie in het model.


In [6]:
# ── Cel 04b · Diagnose tijdreeksgaten ────────────────────────────────

print("Startdatum per parking in train:")
start_check = (
    train.groupby("parking_id")["rounded_hour"]
    .agg(["min", "max", "count"])
    .rename(columns={"min":"start","max":"einde","count":"n_rijen"})
)
print(start_check.to_string())

print("\nVerwacht aaneengesloten uurdata per parking:")
for pid, row in start_check.iterrows():
    verwacht = int((row["einde"] - row["start"]).total_seconds() / 3600) + 1
    werkelijk = row["n_rijen"]
    gat = verwacht - werkelijk
    status = "✅" if gat == 0 else f"⚠ GAT van {gat} uur"
    print(f"  {pid:<15} verwacht={verwacht:>6} | werkelijk={werkelijk:>6} | {status}")


Startdatum per parking in train:
                            start               einde  n_rijen
parking_id                                                    
P Bruul       2020-01-01 00:00:00 2024-12-31 23:00:00    12709
P Grote Markt 2020-01-01 00:00:00 2024-12-31 23:00:00    19044
P Hoogstraat  2023-01-01 00:00:00 2024-12-31 23:00:00    12954
P Kathedraal  2020-01-01 21:00:00 2024-12-31 23:00:00    12232
P Keerdok     2023-01-17 00:00:00 2024-12-31 23:00:00    16810
P Komet       2024-08-23 01:00:00 2024-12-31 23:00:00     3126
P Lamot       2020-01-01 00:00:00 2024-12-31 23:00:00    14013
P Maarten     2024-04-26 16:00:00 2024-12-31 23:00:00     5944
P Tinel       2020-01-01 00:00:00 2024-12-31 23:00:00    18155
P Veemarkt    2020-01-01 00:00:00 2024-12-31 23:00:00    14850

Verwacht aaneengesloten uurdata per parking:
  P Bruul         verwacht= 43848 | werkelijk= 12709 | ⚠ GAT van 31139 uur
  P Grote Markt   verwacht= 43848 | werkelijk= 19044 | ⚠ GAT van 24804 uur
  P Hoogstraat 

## Stap 3c — Tijdsbewuste lag-features via merge

De trainingsset bevat substantiële tijdreeksgaten als gevolg van de 
kwaliteitsfilter uit nb05 (low_data_coverage, system_blackout, 
partial_year). Een naïeve `shift(n)`-operatie over rij-positie 
produceert in dit geval valse lagwaarden: de "vorige uurwaarde" 
kan in werkelijkheid dagen of weken oud zijn.

De correcte aanpak is een **tijdsbewuste lag via self-join**: 
voor elke observatie op tijdstip t zoeken we expliciet de observatie 
op tijdstip t − k uur op. Als die niet bestaat (gat in de reeks), 
krijgt de lag-kolom de waarde NaN — wat correct is. Dit principe 
wordt beschreven in Cerqueira et al. (2023) als vereiste voor 
betrouwbare autoregressive features in onderbroken tijdreeksen.

**Consequentie voor nb09:** alle rijen met NaN in een lag-feature 
worden gedropped vóór modeltraining. Gezien de omvang van de gaten 
zal dit de effectieve trainingsset substantieel verkleinen. Dit wordt 
gerapporteerd als een expliciete datakwaliteitsbeperking.


In [7]:
# ── Cel 04c · Tijdsbewuste lag-features via merge ─────────────────────
# Verwijder de foutief berekende lag-kolommen eerst
for df in [train, holdout]:
    for col in ["occ_lag_1h", "occ_lag_24h", "occ_lag_168h"]:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)

def add_time_aware_lags(df, lag_hours_list):
    """
    Voegt lag-features toe via tijdsbewuste self-join.
    Lag is enkel geldig als er een observatie bestaat op exact t - k uur.
    Ontbrekende observaties (tijdreeksgaten) krijgen NaN.
    """
    base = df[["parking_id", "rounded_hour", "occupancy_rate"]].copy()

    for k in lag_hours_list:
        lag_src = base.copy()
        lag_src["rounded_hour"] = (
            lag_src["rounded_hour"] + pd.Timedelta(hours=k)
        )
        lag_src = lag_src.rename(
            columns={"occupancy_rate": f"occ_lag_{k}h"})

        df = df.merge(
            lag_src[["parking_id", "rounded_hour", f"occ_lag_{k}h"]],
            on=["parking_id", "rounded_hour"],
            how="left"
        )
    return df

train   = add_time_aware_lags(train,   [1, 24, 168])
holdout = add_time_aware_lags(holdout, [1, 24, 168])

# ── NaN-rapport na tijdsbewuste lag ───────────────────────────────────
print("NaN-rapport tijdsbewuste lag-features:\n")
print(f"{'Feature':<16} {'Train NaN':>10} {'Train %':>8} "
      f"{'Holdout NaN':>12} {'Holdout %':>10}")
for col in ["occ_lag_1h", "occ_lag_24h", "occ_lag_168h"]:
    nt  = train[col].isnull().sum()
    pct_t = nt / len(train) * 100
    nh  = holdout[col].isnull().sum()
    pct_h = nh / len(holdout) * 100
    print(f"{col:<16} {nt:>10} {pct_t:>7.1f}% {nh:>12} {pct_h:>9.1f}%")

print(f"\nTrain rijen met ALLE 3 lags aanwezig: "
      f"{train[['occ_lag_1h','occ_lag_24h','occ_lag_168h']].notna().all(axis=1).sum():,}"
      f" / {len(train):,} "
      f"({train[['occ_lag_1h','occ_lag_24h','occ_lag_168h']].notna().all(axis=1).mean()*100:.1f}%)")

print(f"Holdout rijen met ALLE 3 lags aanwezig: "
      f"{holdout[['occ_lag_1h','occ_lag_24h','occ_lag_168h']].notna().all(axis=1).sum():,}"
      f" / {len(holdout):,} "
      f"({holdout[['occ_lag_1h','occ_lag_24h','occ_lag_168h']].notna().all(axis=1).mean()*100:.1f}%)")


NaN-rapport tijdsbewuste lag-features:

Feature           Train NaN  Train %  Holdout NaN  Holdout %
occ_lag_1h             7494     5.8%           10       0.0%
occ_lag_24h            9620     7.4%          240       0.3%
occ_lag_168h          11890     9.2%         1680       1.9%

Train rijen met ALLE 3 lags aanwezig: 108,519 / 129,837 (83.6%)
Holdout rijen met ALLE 3 lags aanwezig: 85,920 / 87,600 (98.1%)


## Stap 3 — Conclusie lag-features

De tijdsbewuste self-join produceert de volgende bruikbare trainingsset:

| Lag | Train NaN | Train % | Interpretatie |
|-----|-----------|---------|---------------|
| `occ_lag_1h`   | 7.494  | 5.8% | Gaten < 1 uur in tijdreeks |
| `occ_lag_24h`  | 9.620  | 7.4% | Gaten < 24 uur |
| `occ_lag_168h` | 11.890 | 9.2% | Gaten < 168 uur |
| **Alle 3 aanwezig** | — | **83.6%** | **Effectieve trainingsset nb09** |

De 16.4% uitval (21.318 rijen) is een directe consequentie van de 
kwaliteitsfilter uit nb05 en wordt gerapporteerd als een expliciete 
datakwaliteitsbeperking in de thesis. De holdout (98.1% volledig) 
is nagenoeg onaangetast — evaluatiemetrieken in nb11 zijn dus 
representatief voor de volledige 2025-periode.

> **Nb09-instructie:** drop alle rijen waar `occ_lag_168h` NaN is 
> vóór modeltraining. Gebruik `occ_lag_168h` als de meest 
> conservatieve NaN-mask — wie lag_168h heeft, heeft ook lag_1h 
> en lag_24h (strikt superset).


## Stap 4 — Weersfeatures

Drie weersfeatures worden opgenomen op basis van de FASE02-hypothesentoetsen.
`temp_c` wordt **niet** opgenomen: de post-hoc Simpson's Paradox analyse 
(nb07) toonde een globale ρ = −0.013 en een partiële ρ = −0.050 na correctie 
voor daglengte — het seizoenspatroon wordt volledig gevangen door 
`month_sin/cos`.

### 4.1 Neerslag — drempelpatroon via binning

H-E1 toonde een niet-lineair drempelpatroon: het effect van neerslag op 
bezetting is niet proportioneel maar categorisch (geen neerslag vs. 
lichte neerslag is de kritische overgang). Binning op RMI-categorieën 
is daarom superieur aan een continue representatie (Oz, 2023).

Grenzen: 0 mm/u = droog | 0–2 mm/u = licht | 2–5 mm/u = matig | >5 mm/u = zwaar

### 4.2 Wind — binaire drempel

H-E6 toonde een mediaan bezettingsstijging van +5.3% bij wind > 10 m/s, 
consistent met modal shift van fiets naar auto bij sterke wind. Een 
binaire drempel is voldoende gezien het drempelkarakter van het effect.

### 4.3 Zonneschijnduur — gestandaardiseerd continu

H-E7 werd zwak verworpen (ρ = +0.081), maar zonneschijnduur is 
opgenomen als lage-prioriteit feature gezien de beperkte collineariteit 
met de seizoensencoding. StandardScaler wordt uitsluitend op de 
trainingsset gefit.


In [8]:
# ── Cel 05 · Weersfeatures ────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
import pandas as pd

# ── 4.1 Neerslag binning (RMI-categorieën) ───────────────────────────
# Vaste grenzen — geen fitting, geen leakage-risico
precip_bins   = [-0.001, 0.0, 2.0, 5.0, float("inf")]
precip_labels = ["droog", "licht", "matig", "zwaar"]

for df in [train, holdout]:
    df["precip_bin"] = pd.cut(
        df["precip_mm"],
        bins=precip_bins,
        labels=precip_labels
    ).astype(str)

# ── 4.2 Wind drempel-dummy ────────────────────────────────────────────
for df in [train, holdout]:
    df["wind_strong"] = (df["wind_speed_ms"] > 10).astype(int)

# ── 4.3 Zonneschijnduur — StandardScaler ─────────────────────────────
# FIT uitsluitend op train — transform op beide
scaler_sun = StandardScaler()
train["sun_duration_scaled"]   = scaler_sun.fit_transform(
    train[["sun_duration_min"]])
holdout["sun_duration_scaled"] = scaler_sun.transform(
    holdout[["sun_duration_min"]])

# ── Verificatie ───────────────────────────────────────────────────────
print("── 4.1 precip_bin verdeling (train) ──")
print(train["precip_bin"].value_counts().sort_index())

print("\n── 4.2 wind_strong (train) ──")
print(f"  wind > 10 m/s : {train['wind_strong'].sum():,} rijen "
      f"({train['wind_strong'].mean()*100:.1f}%)")

print("\n── 4.3 sun_duration_scaled (train) ──")
print(f"  Scaler mean : {scaler_sun.mean_[0]:.2f} min")
print(f"  Scaler std  : {scaler_sun.scale_[0]:.2f} min")
print(f"  Scaled range: [{train['sun_duration_scaled'].min():.2f}, "
      f"{train['sun_duration_scaled'].max():.2f}]")
print(f"  Holdout range (ongefitst): [{holdout['sun_duration_scaled'].min():.2f}, "
      f"{holdout['sun_duration_scaled'].max():.2f}]")
print(f"\n  ⚠ Holdout buiten train range? "
      f"{'JA — rapporteren als kanttekening' if holdout['sun_duration_scaled'].max() > train['sun_duration_scaled'].max() else 'Nee ✅'}")


── 4.1 precip_bin verdeling (train) ──
precip_bin
droog    109852
licht     18338
matig      1337
zwaar       310
Name: count, dtype: int64

── 4.2 wind_strong (train) ──
  wind > 10 m/s : 197 rijen (0.2%)

── 4.3 sun_duration_scaled (train) ──
  Scaler mean : 12.68 min
  Scaler std  : 22.01 min
  Scaled range: [-0.58, 2.15]
  Holdout range (ongefitst): [-0.58, 2.15]

  ⚠ Holdout buiten train range? Nee ✅


## Stap 4 — Conclusie weersfeatures

| Feature | Verdeling | Status |
|---|---|---|
| `precip_bin` | droog=84.6%, licht=14.1%, matig=1.0%, zwaar=0.2% | ✅ Correct |
| `wind_strong` | positief=0.2% (197 rijen) | ⚠ Zie hieronder |
| `sun_duration_scaled` | range [-0.58, 2.15], holdout binnen train-range | ✅ Correct |

### ⚠ Kritische noot: `wind_strong` is quasi-leeg

Met slechts 197 positieve rijen (0.2% van de trainingsset) is 
`wind_strong` een extreem spaarzame feature. Boomgebaseerde modellen 
(Random Forest, XGBoost) kunnen zulke features principieel leren, 
maar de schattingsonzekerheid is hoog door het kleine aantal gevallen. 
De feature wordt meegenomen in de VIF-check en permutatie-importance 
in nb11 — als de importance verwaarloosbaar blijkt, wordt ze 
gedropped vóór de finale modelversie. Dit wordt expliciet gerapporteerd.

De `precip_bin`-verdeling is realistisch voor het Belgische klimaat: 
84.6% droge uren is consistent met RMI-klimatologische normen voor 
Mechelen (~320 neerslagdagen per jaar maar lage intensiteit per uur).


## Stap 5 — Ruimtelijke features

Drie ruimtelijke features worden aangemaakt. Ze vertegenwoordigen 
elk een ander niveau van ruimtelijke informatie:

| Feature | Niveau | Basis |
|---|---|---|
| `parking_id_encoded` | Parking-specifiek | Target encoding — gemiddelde bezetting per parking |
| `cluster_data` | Gedragscluster | Data-gedreven k=2, silhouette=0.615 (nb06) |
| `high_lt_pressure` | Structureel plafond | LT-ceiling effect P Hoogstraat + P Komet (nb07b) |

### Target encoding `parking_id`

Met 10 parkings is one-hot encoding technisch haalbaar (10 dummy-kolommen), 
maar target encoding is empirisch superieur voor boomgebaseerde modellen: 
het comprimeert parking-identiteit naar één informatiedense kolom die de 
gemiddelde bezetting per parking representeert (Pargent et al., 2022). 
Scikit-learn's `TargetEncoder` gebruikt interne k-fold cross-validation 
(cv=5) om target leakage te vermijden — de geëncodeerde waarde per rij 
is nooit berekend op basis van die rij zelf.

**Fit uitsluitend op train — transform op beide.**

### `cluster_data`

De data-gedreven clusteroplossing uit nb06 (k=2, silhouette=0.615) 
wees aan dat drie parkings gedragsmatig afwijken van hun administratieve 
tier. `cluster_data` vangt deze gedragsmatige scheiding op en is 
complementair aan `tier_num`.

Cluster 0 (centrumgedrag): P Grote Markt, P Kathedraal, P Lamot, 
P Veemarkt, P Keerdok, P Tinel  
Cluster 1 (vestengedrag): P Bruul, P Komet, P Maarten, P Hoogstraat


In [9]:
# ── Cel 06 · Ruimtelijke features ─────────────────────────────────────
from sklearn.preprocessing import TargetEncoder

# ── 5.1 cluster_data (hard-coded uit nb06) ────────────────────────────
cluster_map = {
    "P Grote Markt": 0, "P Kathedraal":  0, "P Lamot":   0,
    "P Veemarkt":    0, "P Keerdok":     0, "P Tinel":   0,
    "P Bruul":       1, "P Komet":       1, "P Maarten": 1,
    "P Hoogstraat":  1
}
for df in [train, holdout]:
    df["cluster_data"] = df["parking_id"].map(cluster_map)

# ── 5.2 high_lt_pressure (structureel LT-plafond) ────────────────────
for df in [train, holdout]:
    df["high_lt_pressure"] = (
        df["parking_id"].isin(["P Hoogstraat", "P Komet"])
    ).astype(int)

# ── 5.3 Target encoding parking_id ───────────────────────────────────
# FIT op train only — cv=5 voorkomt target leakage intern
te = TargetEncoder(cv=5, smooth="auto", random_state=42)
train["parking_id_encoded"]   = te.fit_transform(
    train[["parking_id"]], train["occupancy_rate"]).ravel()
holdout["parking_id_encoded"] = te.transform(
    holdout[["parking_id"]]).ravel()

# ── Verificatie ───────────────────────────────────────────────────────
print("── 5.1 cluster_data ──")
print(train.groupby(["parking_id","cluster_data","tier_admin"])
      .size().reset_index(name="n").to_string(index=False))

print("\n── 5.2 high_lt_pressure ──")
print(train.groupby(["parking_id","high_lt_pressure"])
      .size().reset_index(name="n")
      [["parking_id","high_lt_pressure","n"]].to_string(index=False))

print("\n── 5.3 Target encoding parking_id ──")
enc_check = (
    train.groupby("parking_id")
    .agg(mean_occ=("occupancy_rate","mean"),
         te_encoded=("parking_id_encoded","mean"))
    .round(4)
    .sort_values("mean_occ")
)
print(enc_check.to_string())
print(f"\nCorrelatie mean_occ vs. te_encoded: "
      f"{enc_check['mean_occ'].corr(enc_check['te_encoded']):.4f} "
      f"(moet ≈ 1.0 zijn)")


── 5.1 cluster_data ──
   parking_id  cluster_data     tier_admin     n
      P Bruul             1 vesten_of_rand 12709
P Grote Markt             0        centrum 19044
 P Hoogstraat             1        centrum 12954
 P Kathedraal             0        centrum 12232
    P Keerdok             0 vesten_of_rand 16810
      P Komet             1 vesten_of_rand  3126
      P Lamot             0        centrum 14013
    P Maarten             1 vesten_of_rand  5944
      P Tinel             0 vesten_of_rand 18155
   P Veemarkt             0        centrum 14850

── 5.2 high_lt_pressure ──
   parking_id  high_lt_pressure     n
      P Bruul                 0 12709
P Grote Markt                 0 19044
 P Hoogstraat                 1 12954
 P Kathedraal                 0 12232
    P Keerdok                 0 16810
      P Komet                 1  3126
      P Lamot                 0 14013
    P Maarten                 0  5944
      P Tinel                 0 18155
   P Veemarkt                 

## Stap 6 — Event-features en interactietermen

De event-dummies `is_football_day`, `is_festival_day`, `is_kermis_day`,  
`is_carnival_day` en `is_procession_day` bestaan reeds in de dataset.  
De generieke `is_event_day` wordt **niet** als zelfstandige feature  
opgenomen — deze maskeert tegengestelde tier-effecten die empirisch  
zijn vastgesteld (Wilcoxon matched pairs, nb07).

### Interactietermen

Uit de Wilcoxon-analyse (nb07) bleken voetbal en festival tegengestelde  
effecten te hebben per tier (substitutiepatroon). Een enkelvoudige  
event-dummy zonder tier-interactie zou deze tegengestelde effecten  
middelen naar nul — waardoor het model een cruciale structuur mist.

De interactieterm is het rekenkundig product van twee binaire kolommen  
en vereist geen fitting:

$$\text{voetbal\_x\_centrum} = \text{is\_football\_day} \times \text{tier\_num}$$

Deze term is 1 uitsluitend wanneer het tegelijk een voetbaldag én  
een centrumparking is — precies de combinatie waar het negatieve  
substitutie-effect optreedt.

### Cascade-features

De t-toets analyse (H-E3, nb07) toonde significante bezettingseffecten  
op t−1 en t+1 rondom events. `hours_to_next_event` en  
`hours_since_last_event` modelleren deze temporele cascade als  
continue features, wat superieur is aan binaire dag-dummies voor  
het vangen van graduele aanloop- en naijleffecten.


In [10]:
# ── Cel 07 · Event-interacties en cascade-features ────────────────────

# ── 6.1 Interactietermen ─────────────────────────────────────────────
for df in [train, holdout]:
    df["voetbal_x_centrum"]  = df["is_football_day"].astype(int)  * df["tier_num"]
    df["festival_x_centrum"] = df["is_festival_day"].astype(int)  * df["tier_num"]
    df["carnaval_x_centrum"] = df["is_carnival_day"].astype(int)  * df["tier_num"]
    df["vakantie_x_centrum"] = df["is_school_vacation"].astype(int) * df["tier_num"]
    df["feestdag_x_centrum"] = df["is_national_holiday"].astype(int) * df["tier_num"]

# ── 6.2 Cascade-features ─────────────────────────────────────────────
def compute_cascade_features(df):
    df = df.sort_values(["parking_id", "rounded_hour"]).copy()
    event_flag = df["is_event_day"].astype(bool)

    hours_since = []
    hours_to    = []

    for pid, group in df.groupby("parking_id", sort=False):
        flags = group["is_event_day"].astype(bool).values
        n     = len(flags)

        # hours_since_last_event
        since = np.full(n, np.nan)
        last  = np.nan
        for i in range(n):
            if flags[i]:
                last = 0
            elif not np.isnan(last):
                last += 1
            since[i] = last

        # hours_to_next_event
        to_next = np.full(n, np.nan)
        nxt     = np.nan
        for i in range(n - 1, -1, -1):
            if flags[i]:
                nxt = 0
            elif not np.isnan(nxt):
                nxt += 1
            to_next[i] = nxt

        hours_since.extend(since)
        hours_to.extend(to_next)

    df["hours_since_last_event"] = hours_since
    df["hours_to_next_event"]    = hours_to
    return df

train   = compute_cascade_features(train)
holdout = compute_cascade_features(holdout)

# ── Verificatie ───────────────────────────────────────────────────────
print("── 6.1 Interactietermen — positieve gevallen (train) ──")
interact_cols = ["voetbal_x_centrum","festival_x_centrum",
                 "carnaval_x_centrum","vakantie_x_centrum","feestdag_x_centrum"]
for c in interact_cols:
    n_pos = train[c].sum()
    print(f"  {c:<25} n_positief = {n_pos:>6,} ({n_pos/len(train)*100:.1f}%)")

print("\n── 6.2 Cascade-features (train) ──")
for c in ["hours_since_last_event","hours_to_next_event"]:
    n_null = train[c].isnull().sum()
    print(f"  {c:<28} NaN={n_null:,} | "
          f"min={train[c].min():.0f} | "
          f"max={train[c].max():.0f} | "
          f"median={train[c].median():.0f}")


── 6.1 Interactietermen — positieve gevallen (train) ──
  voetbal_x_centrum         n_positief =  2,156 (1.7%)
  festival_x_centrum        n_positief =    157 (0.1%)
  carnaval_x_centrum        n_positief =    807 (0.6%)
  vakantie_x_centrum        n_positief = 17,443 (13.4%)
  feestdag_x_centrum        n_positief =  2,463 (1.9%)

── 6.2 Cascade-features (train) ──
  hours_since_last_event       NaN=15,754 | min=0 | max=1728 | median=166
  hours_to_next_event          NaN=17,248 | min=0 | max=4199 | median=165


## Stap 5–6 — Conclusie ruimtelijke features & event-interacties

**Ruimtelijke features ✅**  
Target encoding correct: correlatie mean_occ vs. te_encoded = 1.0000.  
De minimale afwijking bij P Lamot (0.4497 vs. 0.4496) is precies het  
verwachte effect van de cv=5 smoothing bij grote groepen — correct.

**Interactietermen ✅**  
`vakantie_x_centrum` heeft de hoogste densiteit (13.4%) — consistent  
met de sterke H-E8 bevinding. Voetbal (1.7%) en feestdagen (1.9%)  
zijn spaarzaam maar voldoende voor boomgebaseerde modellen.

**Cascade-features ⚠ — twee actiepunten:**

1. `hours_to_next_event` max = 4.199 uur (≈175 dagen) — dit is  
   realistisch voor P Komet die pas start in augustus 2024 met  
   weinig events daarna, maar het signaleert dat de feature een  
   zeer brede range heeft.

2. NaN = 15.754–17.248 rijen (12–13%): dit zijn rijen vóór het  
   eerste event (hours_since) en ná het laatste event (hours_to)  
   per parking. Deze worden geïmputeerd met een grote schildwachtwaarde  
   (999) die het model signaleert: "ver van enig event". Droppen is  
   geen optie — het zou 13% van de trainingsset elimineren bovenop  
   de al-bestaande lag-NaN uitval.


In [11]:
# ── Cel 07b · Cascade NaN-imputatie met schildwachtwaarde ─────────────

SENTINEL = 999  # uur — "ver van enig event"

for df in [train, holdout]:
    df["hours_since_last_event"] = (
        df["hours_since_last_event"].fillna(SENTINEL)
    )
    df["hours_to_next_event"] = (
        df["hours_to_next_event"].fillna(SENTINEL)
    )

# Clip bovengrens op sentinel (max=4199 → 999 na clip)
for df in [train, holdout]:
    df["hours_since_last_event"] = df["hours_since_last_event"].clip(upper=SENTINEL)
    df["hours_to_next_event"]    = df["hours_to_next_event"].clip(upper=SENTINEL)

# Verificatie
print("Cascade-features na imputatie (train):")
for c in ["hours_since_last_event", "hours_to_next_event"]:
    print(f"  {c:<28} NaN={train[c].isnull().sum()} | "
          f"min={train[c].min():.0f} | "
          f"max={train[c].max():.0f} | "
          f"median={train[c].median():.0f}")


Cascade-features na imputatie (train):
  hours_since_last_event       NaN=0 | min=0 | max=999 | median=214
  hours_to_next_event          NaN=0 | min=0 | max=999 | median=218


## Stap 7 — Kalenderfeatures afronden

Twee kleine maar noodzakelijke stappen:

1. `year_2020`: categorische dummy die het structurele COVID-shift  
   effect vangt (H-T4: r_struct = −0.956 voor centrumtier,  
   Δ = +18.6% voor vesten). **Niet ordinaal** — 2020 is kwalitatief  
   anders dan 2023/2024, niet "kleiner".

2. `is_school_vacation` en `is_national_holiday` zijn momenteel  
   `float64` (waarden 0.0/1.0) — casten naar `int8` voor consistentie  
   met de overige binaire features en efficiënter geheugengebruik.


In [12]:
# ── Cel 08 · Kalenderfeatures afronden ───────────────────────────────

for df in [train, holdout]:
    # year_2020: categorisch — 2020 vs. alle andere jaren
    df["year_2020"] = (df["year"] == 2020).astype(int)

    # Float → int8 voor binaire kalenderkolommen
    for col in ["is_school_vacation", "is_national_holiday",
                "is_other_holiday",   "is_any_holiday"]:
        df[col] = df[col].astype(int)

# Verificatie
print("year_2020 verdeling:")
print(f"  Train   — 2020: {train['year_2020'].sum():,} | "
      f"overig: {(train['year_2020']==0).sum():,}")
print(f"  Holdout — 2020: {holdout['year_2020'].sum():,} | "
      f"overig: {(holdout['year_2020']==0).sum():,}")

print("\nDtype-check kalenderkolommen (train):")
for col in ["is_school_vacation","is_national_holiday","year_2020"]:
    print(f"  {col:<25} dtype={train[col].dtype} | "
          f"uniek={sorted(train[col].unique())}")


year_2020 verdeling:
  Train   — 2020: 32,741 | overig: 97,096
  Holdout — 2020: 0 | overig: 87,600

Dtype-check kalenderkolommen (train):
  is_school_vacation        dtype=int64 | uniek=[np.int64(0), np.int64(1)]
  is_national_holiday       dtype=int64 | uniek=[np.int64(0), np.int64(1)]
  year_2020                 dtype=int64 | uniek=[np.int64(0), np.int64(1)]


## Stap 8 — Finale feature-matrix: definiëren, VIF-check en export

### Definitieve feature-kolommen

De volledige feature-matrix bevat 32 modelfeatures + 4 metadata-kolommen.
Metadata wordt mee geëxporteerd voor stratificatie in nb11/nb12 maar  
**niet** als modelinput gebruikt.

### VIF-check

Variance Inflation Factor detecteert multicollineariteit tussen features.  
VIF = 1 betekent geen correlatie; VIF > 10 is de gangbare drempelwaarde  
voor problematische multicollineariteit (Hair et al., 2019). Interactie-  
termen zijn per constructie gecorreleerd met hun componentfeatures —  
VIF > 5 voor interacties is aanvaardbaar en verwacht.


In [13]:
# ── Cel 09 · Finale feature-matrix definiëren ─────────────────────────

FEATURE_COLS = [
    # Temporeel cyclisch (6)
    "hour_sin", "hour_cos",
    "weekday_sin", "weekday_cos",
    "month_sin", "month_cos",
    # Kalender (4)
    "is_school_vacation", "is_national_holiday",
    "is_any_holiday", "year_2020",
    # Lag-features (3)
    "occ_lag_1h", "occ_lag_24h", "occ_lag_168h",
    # Weer (3)
    "wind_strong", "sun_duration_scaled",
    # Ruimtelijk (4)
    "parking_id_encoded", "tier_num",
    "cluster_data", "high_lt_pressure",
    # Events (5)
    "is_football_day", "is_festival_day", "is_kermis_day",
    "is_carnival_day", "is_procession_day",
    # Interacties (5)
    "voetbal_x_centrum", "festival_x_centrum", "carnaval_x_centrum",
    "vakantie_x_centrum", "feestdag_x_centrum",
    # Cascade (2)
    "hours_since_last_event", "hours_to_next_event",
]

# precip_bin wordt apart one-hot gecodeerd → 3 extra kolommen (ref=droog)
# day_type_3 wordt apart one-hot gecodeerd → 2 extra kolommen (ref=weekday)

METADATA_COLS = ["rounded_hour", "parking_id", "tier_admin",
                 "year", "occupancy_rate"]

print(f"Aantal modelfeatures (voor one-hot): {len(FEATURE_COLS)}")
print(f"  + precip_bin one-hot:  +3 kolommen (licht/matig/zwaar)")
print(f"  + day_type_3 one-hot:  +2 kolommen (saturday/sunday_holiday)")
print(f"  = Totaal na one-hot:   {len(FEATURE_COLS) + 5} features")
print(f"\nMetadata-kolommen: {METADATA_COLS}")

# ── Dtype-check: alle features moeten numeriek zijn ───────────────────
print("\nDtype-check FEATURE_COLS (train) — strings/categoricals zijn fout:")
probleem = []
for col in FEATURE_COLS:
    dtype = str(train[col].dtype)
    if "str" in dtype or "object" in dtype or "category" in dtype:
        probleem.append(col)
        print(f"  ⚠ {col:<30} dtype={dtype}")
    else:
        pass  # stil voor correcte kolommen

if not probleem:
    print("  ✅ Alle feature-kolommen zijn numeriek")
else:
    print(f"\n  {len(probleem)} kolom(men) vereisen conversie vóór export")


Aantal modelfeatures (voor one-hot): 31
  + precip_bin one-hot:  +3 kolommen (licht/matig/zwaar)
  + day_type_3 one-hot:  +2 kolommen (saturday/sunday_holiday)
  = Totaal na one-hot:   36 features

Metadata-kolommen: ['rounded_hour', 'parking_id', 'tier_admin', 'year', 'occupancy_rate']

Dtype-check FEATURE_COLS (train) — strings/categoricals zijn fout:
  ✅ Alle feature-kolommen zijn numeriek


In [14]:
# ── Cel 10 · VIF-check (zonder statsmodels) ───────────────────────────
# VIF voor feature j = 1 / (1 - R²_j)
# waarbij R²_j = R² van een regressie van feature j op alle andere features
# Volledig in numpy/sklearn — geen statsmodels nodig

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# One-hot precip_bin en day_type_3 voor VIF
train_vif = pd.get_dummies(
    train[FEATURE_COLS + ["precip_bin", "day_type_3"]].dropna(),
    columns=["precip_bin", "day_type_3"],
    drop_first=True
).astype(float)

X = train_vif.values
cols = train_vif.columns.tolist()

vif_scores = []
for i, col in enumerate(cols):
    y_j = X[:, i]
    X_rest = np.delete(X, i, axis=1)
    r2 = LinearRegression().fit(X_rest, y_j).score(X_rest, y_j)
    vif = 1 / (1 - r2) if r2 < 1.0 else np.inf
    vif_scores.append((col, round(vif, 2)))

vif_df = (pd.DataFrame(vif_scores, columns=["feature", "VIF"])
          .sort_values("VIF", ascending=False)
          .reset_index(drop=True))

print("VIF-rapport (gesorteerd, hoog → laag):\n")
print(vif_df.to_string(index=False))

print(f"\n── Drempelcheck ──")
print(f"VIF > 10 (problematisch):  "
      f"{vif_df[vif_df['VIF'] > 10]['feature'].tolist()}")
print(f"VIF 5–10 (verhoogd):       "
      f"{vif_df[(vif_df['VIF'] > 5) & (vif_df['VIF'] <= 10)]['feature'].tolist()}")
print(f"VIF ≤ 5  (aanvaardbaar):   "
      f"{(vif_df['VIF'] <= 5).sum()} van {len(vif_df)} features")


VIF-rapport (gesorteerd, hoog → laag):

                  feature   VIF
             cluster_data 11.10
       parking_id_encoded 10.97
       day_type_3_weekday  3.85
      is_national_holiday  3.37
               occ_lag_1h  3.07
             occ_lag_168h  2.95
              occ_lag_24h  2.68
              weekday_sin  2.63
          is_carnival_day  2.61
       carnaval_x_centrum  2.56
       vakantie_x_centrum  2.36
day_type_3_sunday_holiday  2.35
         high_lt_pressure  2.32
          is_football_day  2.30
                 tier_num  2.30
       feestdag_x_centrum  2.28
        voetbal_x_centrum  2.22
       is_school_vacation  2.22
   hours_since_last_event  2.04
      hours_to_next_event  2.01
           is_any_holiday  1.89
      sun_duration_scaled  1.59
                 hour_cos  1.44
                month_cos  1.44
                year_2020  1.44
          is_festival_day  1.43
       festival_x_centrum  1.40
            is_kermis_day  1.27
                month_sin  1.26


## Stap 8 — VIF-interpretatie en beslissing

### `cluster_data` (VIF = 11.10) en `parking_id_encoded` (VIF = 10.97)

Deze twee features zijn hoog gecorreleerd met elkaar — en dat is 
**structureel verwacht en methodologisch verdedigbaar**. Beide 
representeren parking-identiteit op een ander abstractieniveau:

- `parking_id_encoded`: continue representatie van de gemiddelde 
  bezetting per parking (parking-specifiek signaal)
- `cluster_data`: binaire gedragscluster op basis van uurprofiel 
  (nb06, silhouette = 0.615)

Ze zijn gecorreleerd omdat parkings met hoge gemiddelde bezetting 
voornamelijk in cluster 0 zitten (P Tinel, P Veemarkt, P Kathedraal 
etc.). De VIF-drempel van 10 is echter een heuristiek ontwikkeld 
voor **lineaire regressie** waarbij multicollineariteit de 
coëfficiëntschattingen destabiliseert. Voor **boomgebaseerde modellen** 
(Random Forest, XGBoost — het doelmodel in nb09) is multicollineariteit 
geen probleem: bomen splitsen op één feature tegelijk en zijn 
immuun voor colineariteitsbias (Hastie et al., 2009).

**Beslissing: beide features worden behouden.**  
Motivatie: ze meten complementaire ruimtelijke aspecten 
(gemiddeld bezettingsniveau vs. gedragsprofiel). In nb12 (SHAP) 
zal de relatieve importance empirisch bepalen welke van beide 
meer informatief is — als één van beide verwaarloosbare SHAP-waarden 
heeft, wordt ze pas dán gedropped als post-hoc vereenvoudiging.

> *"VIF-waarden > 10 voor `cluster_data` en `parking_id_encoded` 
> reflecteren de structurele correlatie tussen gemiddeld 
> bezettingsniveau en gedragsclusterlidmaatschap. Aangezien het 
> predictiemodel boomgebaseerd is (Random Forest / XGBoost), 
> introduceert multicollineariteit geen schattingsbias; beide 
> features worden behouden als complementaire ruimtelijke 
> representaties (Hastie et al., 2009)."*

### Overige bevindingen ✅
- `is_any_holiday` (VIF = 1.89) en `is_national_holiday` (VIF = 3.37) 
  coëxisteren — dit is aanvaardbaar maar `is_any_holiday` is een 
  superset van `is_national_holiday`. We behouden beide omdat 
  `is_any_holiday` ook andere feestdagen (brugdagen etc.) omvat.
- Cascade-features: NaN = 0 na imputatie ✅, max = 999 (sentinel) ✅
- Holdout bevat geen 2020-rijen: `year_2020` = 0 voor alle holdout 
  rijen ✅ — correct, dit is de out-of-sample 2025-periode.


## Stap 8b — Aanvullingen op de feature-matrix

Vóór de export worden twee bijkomende features toegevoegd en twee 
kolommen hernoemd op basis van een kritische review van de initiële 
feature-matrix. Een derde kandidaat-feature (`hours_to_kickoff`) 
wordt expliciet verworpen.

---

### Toevoeging 1: `log_capacity`

De feature `parking_id_encoded` (hernoemd: `mean_occ_by_parking`) 
representeert de gemiddelde historische bezettingsgraad per parking 
en bevat daardoor impliciet partiële capaciteitsinformatie. 
`log_capacity` is echter een conceptueel onderscheiden grootheid: 
het meet de absolute omvang van de parkeerinfrastructuur, 
onafhankelijk van het gebruikspatroon.

De literatuur toont dat parkingcapaciteit een zelfstandige predictor 
is van bezettingsdynamiek: grote parkings vertonen meer buffercapaciteit 
en daardoor vlakkere bezettingscurves, terwijl kleine parkings sneller 
satureren en grotere relatieve schommelingen vertonen (Caicedo, 2010). 
P Keerdok (516 plaatsen) en P Maarten (~50 plaatsen) illustreren dit 
contrast in de Mechelse dataset: beiden hebben een lage-tot-middelmatige 
gemiddelde bezettingsgraad, maar om structureel verschillende redenen.

Een logaritmische transformatie wordt toegepast conform de rechtse 
scheefheid van capaciteitsverdelingen in stedelijke parkingstudies 
(Lira et al., 2021): het verschil tussen 50 en 100 plaatsen is 
gedragsmatig relevanter dan het verschil tussen 450 en 500 plaatsen.

> *"Parkingcapaciteit wordt opgenomen als log-getransformeerde 
> feature (`log_capacity = log(1 + total_capacity)`) op basis van 
> de empirisch vastgestelde niet-lineaire relatie tussen 
> infrastructuuromvang en bezettingsdynamiek (Caicedo, 2010; 
> Lira et al., 2021). De feature is complementair aan de 
> target-geëncodeerde parkingidentiteit (`mean_occ_by_parking`) 
> die het gemiddeld bezettingsniveau representeert."*

---

### Toevoeging 2: `n_concurrent_events`

De event-dummies in de feature-matrix behandelen elk event-type 
afzonderlijk. Mechelen organiseert echter regelmatig overlappende 
events — combinaties zoals kermis én voetbalmatch op dezelfde dag 
zijn aanwezig in `event_names_combined`. De hypothese dat gelijktijdige 
events een super-additief effect hebben op parkeerdruk is inhoudelijk 
plausibel: elke afzonderlijke event-dummy zou het gecombineerde effect 
onderschatten.

`n_concurrent_events` (0, 1 of 2 in de dataset) is reeds aanwezig 
als `int8`-kolom en vereist enkel een dtype-normalisatie. De feature 
is spaarzaam (n_concurrent > 1 is zeldzaam) maar vormt geen 
multicollineariteitsrisico gezien de lage VIF-verwachting.

> *"Het aantal gelijktijdige events (`n_concurrent_events`) wordt 
> opgenomen als ordinale feature om potentieel super-additieve 
> parkeerdruk bij overlappende evenementen te modelleren. De 
> kolom was reeds beschikbaar in de ruwe dataset en vereist geen 
> bijkomende transformatie."*

---

### Verworpen kandidaat: `hours_to_kickoff`

Een uur-precieze voetbalcascade-feature leek initieel aantrekkelijk 
gezien de sterk tijdsgebonden aard van voetbaleffecten. De feature 
werd echter verworpen om twee redenen:

1. **Informatieredundantie:** `hours_to_next_event` vangt reeds de 
   temporele nabijheid van events op — het onderscheid met een 
   voetbal-specifieke variant is marginaal gegeven dat voetbal 
   slechts 1.7% van de trainingsrijen beslaat.

2. **Imputatieartefact:** `football_kickoff_hour` is 97.1% NaN. 
   Imputatie met een schildwachtwaarde (999) zou een artificieel 
   bimodaal signaal injecteren dat het model zou leren als 
   betekenisvol patroon terwijl het een artefact van de 
   datadekking is.

> *"`hours_to_kickoff` werd verworpen als feature wegens 
> informatieredundantie met `hours_to_next_event` en het 
> risico op artefactgedreven modelpatronen door 97.1% NaN-imputatie 
> in de bronkolom `football_kickoff_hour`."*

---

### Hernoemingen

Twee bestaande kolomnamen worden hernoemd voor academische 
consistentie en leesbaarheid:

| Oude naam | Nieuwe naam | Motivatie |
|---|---|---|
| `parking_id_encoded` | `mean_occ_by_parking` | De naam reflecteert de inhoud: gemiddelde historische bezettingsgraad per parking via target encoding |
| `cluster_data` | `behavioral_cluster` | Verwijst expliciet naar de data-gedreven gedragsclusterindeling uit nb06 |


In [16]:
# ── Cel 10b · Aanvullingen + hernoemingen vóór export ─────────────────

# ── Hernoemingen ─────────────────────────────────────────────────────
for df in [train, holdout]:
    df.rename(columns={
        "parking_id_encoded": "mean_occ_by_parking",
        "cluster_data":       "behavioral_cluster"
    }, inplace=True)

FEATURE_COLS = [
    {"parking_id_encoded": "mean_occ_by_parking",
     "cluster_data":       "behavioral_cluster"}.get(c, c)
    for c in FEATURE_COLS
]

# ── Toevoeging 1: log_capacity ────────────────────────────────────────
for df in [train, holdout]:
    df["log_capacity"] = np.log1p(df["total_capacity"])

# ── Toevoeging 2: n_concurrent_events ────────────────────────────────
for df in [train, holdout]:
    df["n_concurrent_events"] = df["n_concurrent_events"].astype(int)

# FEATURE_COLS updaten
FEATURE_COLS = FEATURE_COLS + ["log_capacity", "n_concurrent_events"]

# ── Verificatie ───────────────────────────────────────────────────────
print("── Hernoemingen ──")
print(f"  mean_occ_by_parking aanwezig : "
      f"{'mean_occ_by_parking' in train.columns}")
print(f"  behavioral_cluster  aanwezig : "
      f"{'behavioral_cluster' in train.columns}")

print("\n── log_capacity (train) ──")
print(train.groupby("parking_id")["log_capacity"]
      .first().sort_values().to_string())

print("\n── n_concurrent_events (train) ──")
print(train["n_concurrent_events"].value_counts().sort_index())

print(f"\n── Totaal features na aanvullingen ──")
print(f"  FEATURE_COLS : {len(FEATURE_COLS)} features")
print(f"  + precip_bin one-hot  : +3")
print(f"  + day_type_3 one-hot  : +2")
print(f"  = Totaal na one-hot   : {len(FEATURE_COLS) + 5} features")


── Hernoemingen ──
  mean_occ_by_parking aanwezig : True
  behavioral_cluster  aanwezig : True

── log_capacity (train) ──
parking_id
P Hoogstraat     4.700480
P Komet          4.828314
P Tinel          4.828314
P Veemarkt       4.867534
P Kathedraal     4.875197
P Grote Markt    5.049856
P Maarten        5.247024
P Lamot          5.545177
P Bruul          5.860786
P Keerdok        6.248043

── n_concurrent_events (train) ──
n_concurrent_events
0    112409
1     17320
2       108
Name: count, dtype: int64

── Totaal features na aanvullingen ──
  FEATURE_COLS : 33 features
  + precip_bin one-hot  : +3
  + day_type_3 one-hot  : +2
  = Totaal na one-hot   : 38 features


In [17]:
# ── Cel 11 · Export finale feature-matrix ────────────────────────────

EXPORT_FEATURES = FEATURE_COLS + ["precip_bin", "day_type_3"]
EXPORT_ALL      = EXPORT_FEATURES + METADATA_COLS

train_export   = train[EXPORT_ALL].copy()
holdout_export = holdout[EXPORT_ALL].copy()

# ── Finale NaN-check ─────────────────────────────────────────────────
print("── Finale NaN-check vóór export ──")
nan_train   = train_export[EXPORT_FEATURES].isnull().sum()
nan_holdout = holdout_export[EXPORT_FEATURES].isnull().sum()

# Lag-features apart — verwacht NaN
lag_cols  = ["occ_lag_1h", "occ_lag_24h", "occ_lag_168h"]
rest_cols = [c for c in EXPORT_FEATURES if c not in lag_cols]

prob_train   = nan_train[rest_cols][nan_train[rest_cols] > 0]
prob_holdout = nan_holdout[rest_cols][nan_holdout[rest_cols] > 0]

if prob_train.empty and prob_holdout.empty:
    print("  ✅ Geen onverwachte NaN in niet-lag features")
else:
    print("  ⚠ Onverwachte NaN:")
    print(prob_train)
    print(prob_holdout)

print("\n── Lag-NaN (verwacht — worden gedropped in nb09) ──")
for col in lag_cols:
    nt = train_export[col].isnull().sum()
    nh = holdout_export[col].isnull().sum()
    print(f"  {col:<16} train={nt:>6,} | holdout={nh:>5,}")

# ── Kolom-symmetrie ───────────────────────────────────────────────────
assert list(train_export.columns) == list(holdout_export.columns), \
    "⚠ KOLOMMEN NIET IDENTIEK"
print("\n  ✅ Kolom-symmetrie train == holdout bevestigd")

# ── Export ────────────────────────────────────────────────────────────
train_export.to_parquet(
    DATA_PROC / "train_features.parquet",   index=False)
holdout_export.to_parquet(
    DATA_PROC / "holdout_features.parquet", index=False)

print(f"\n── Shapes ──")
print(f"  train_features  : {train_export.shape[0]:>7,} rijen × "
      f"{train_export.shape[1]} kolommen")
print(f"  holdout_features: {holdout_export.shape[0]:>7,} rijen × "
      f"{holdout_export.shape[1]} kolommen")
print(f"\n✅ Geëxporteerd naar {DATA_PROC}")


── Finale NaN-check vóór export ──
  ✅ Geen onverwachte NaN in niet-lag features

── Lag-NaN (verwacht — worden gedropped in nb09) ──
  occ_lag_1h       train= 7,494 | holdout=   10
  occ_lag_24h      train= 9,620 | holdout=  240
  occ_lag_168h     train=11,890 | holdout=1,680

  ✅ Kolom-symmetrie train == holdout bevestigd

── Shapes ──
  train_features  : 129,837 rijen × 40 kolommen
  holdout_features:  87,600 rijen × 40 kolommen

✅ Geëxporteerd naar /Users/emilevandevoorde/Documents/mechelen_parking/data_processed


In [18]:
# ── Cel 12 · Slotverificatie ──────────────────────────────────────────

train_check   = pd.read_parquet(DATA_PROC / "train_features.parquet")
holdout_check = pd.read_parquet(DATA_PROC / "holdout_features.parquet")

print("=" * 65)
print("  NB08 — FINALE FEATURE-MATRIX OVERZICHT")
print("=" * 65)

categorie_map = {
    "Temporeel cyclisch" : ["hour_sin","hour_cos","weekday_sin",
                            "weekday_cos","month_sin","month_cos"],
    "Kalender"           : ["day_type_3","is_school_vacation",
                            "is_national_holiday","is_any_holiday",
                            "year_2020"],
    "Autoregressive"     : ["occ_lag_1h","occ_lag_24h","occ_lag_168h"],
    "Weer"               : ["precip_bin","wind_strong",
                            "sun_duration_scaled"],
    "Ruimtelijk"         : ["mean_occ_by_parking","behavioral_cluster",
                            "tier_num","high_lt_pressure","log_capacity"],
    "Events"             : ["is_football_day","is_festival_day",
                            "is_kermis_day","is_carnival_day",
                            "is_procession_day","n_concurrent_events"],
    "Interacties"        : ["voetbal_x_centrum","festival_x_centrum",
                            "carnaval_x_centrum","vakantie_x_centrum",
                            "feestdag_x_centrum"],
    "Cascade"            : ["hours_since_last_event",
                            "hours_to_next_event"],
}

total = 0
for cat, cols in categorie_map.items():
    oh_note = " (+3 OH)" if cat == "Weer" else \
              " (+2 OH)" if cat == "Kalender" else ""
    print(f"\n  {cat} ({len(cols)} features{oh_note})")
    for col in cols:
        if col in train_check.columns:
            nn = train_check[col].isnull().sum()
            lag_note = " ← NaN verwacht (nb09 drop)" if "lag" in col else ""
            print(f"    ✅ {col:<32} NaN={nn}{lag_note}")
        else:
            print(f"    ❌ {col:<32} ONTBREEKT")
    total += len(cols)

print(f"\n{'='*65}")
print(f"  Totaal features (vóór one-hot) : {total}")
print(f"  Totaal features (na one-hot)   : {total + 5}")
print(f"  Metadata-kolommen              : {METADATA_COLS}")
print(f"  Target                         : occupancy_rate")
print(f"\n  Train   : {train_check.shape[0]:>7,} rijen")
print(f"  Holdout : {holdout_check.shape[0]:>7,} rijen")
print(f"  Effectieve train na lag-drop   : ~108,519 rijen (83.6%)")
print(f"\n  ✅ Nb08 volledig — klaar voor nb09 (baseline modellering)")
print(f"{'='*65}")


  NB08 — FINALE FEATURE-MATRIX OVERZICHT

  Temporeel cyclisch (6 features)
    ✅ hour_sin                         NaN=0
    ✅ hour_cos                         NaN=0
    ✅ weekday_sin                      NaN=0
    ✅ weekday_cos                      NaN=0
    ✅ month_sin                        NaN=0
    ✅ month_cos                        NaN=0

  Kalender (5 features (+2 OH))
    ✅ day_type_3                       NaN=0
    ✅ is_school_vacation               NaN=0
    ✅ is_national_holiday              NaN=0
    ✅ is_any_holiday                   NaN=0
    ✅ year_2020                        NaN=0

  Autoregressive (3 features)
    ✅ occ_lag_1h                       NaN=7494 ← NaN verwacht (nb09 drop)
    ✅ occ_lag_24h                      NaN=9620 ← NaN verwacht (nb09 drop)
    ✅ occ_lag_168h                     NaN=11890 ← NaN verwacht (nb09 drop)

  Weer (3 features (+3 OH))
    ✅ precip_bin                       NaN=0
    ✅ wind_strong                      NaN=0
    ✅ sun_duration_s

## Nb08 — Methodologische verantwoording feature engineering

### Overzicht

Deze notebook construeerde de definitieve feature-matrix voor de 
tier-gestratificeerde bezettingsmodellen (nb09–nb13). De matrix 
bevat 35 features (40 na one-hot-codering van `precip_bin` en 
`day_type_3`) en is beschikbaar als `train_features.parquet` 
(129.837 rijen) en `holdout_features.parquet` (87.600 rijen).

Alle transformatieparameters werden uitsluitend geschat op de 
trainingsset (2020, 2023, 2024). De holdoutset (2025) onderging 
enkel de corresponderende transformatie, nooit de fitting. 
Dit principe van strikte temporele scheiding voorkomt 
dataleakage en garandeert dat de evaluatiemetrieken in nb11 
een eerlijke out-of-sample schatting van de modelprestatie 
representeren (Cerqueira et al., 2023).

---

### Categorieoverzicht

**Temporeel cyclisch (6 features)**  
Uurlijkse, wekelijkse en maandelijkse periodiciteit worden 
gerepresenteerd via sinusoïdale encoding. Deze aanpak elimineert 
de artificiële discontinuïteit aan de grenzen van elke cyclus 
die ontstaat bij ruwe numerieke tijdsrepresentaties — uur 23 
en uur 0 zijn gedragsmatig aaneengesloten maar numeriek maximaal 
verwijderd (Wan et al., 2023).

**Kalender (5 features, +2 na one-hot)**  
`day_type_3` onderscheidt drie structureel verschillende 
dagprofielen: weekdag, zaterdag (retailpiek) en zondag/feestdag 
(gedrukt profiel). `year_2020` vangt het structurele COVID-shift 
effect op (H-T4: Δ = +18.6% voor vesten, r_struct = −0.956 voor 
centrum) als categorische — niet ordinale — dummy.

**Autoregressive lag-features (3 features)**  
Lags van 1, 24 en 168 uur corresponderen met de drie 
significante ACF-pieken vastgesteld in H-T2. Cruciaal is dat 
de lags werden berekend via tijdsbewuste self-join in plaats 
van positiegebaseerde shift(): de trainingsset bevat 
substantiële tijdreeksgaten (gemiddeld 29% uitval per parking) 
als gevolg van de kwaliteitsfilter in nb05. Een naïeve 
shift()-operatie zou over deze gaten springen en valse 
lagwaarden produceren. De tijdsbewuste methode geeft NaN 
wanneer de doeltijdstempel niet bestaat — methodologisch 
correct maar resulterend in 16.4% rij-uitval in de 
effectieve trainingsset na droppen in nb09.

**Weersfeatures (3 features, +3 na one-hot)**  
`precip_bin` volgt RMI-neerslagcategorieën (droog / licht / 
matig / zwaar) op basis van het drempelpatroon vastgesteld in 
H-E1. `wind_strong` is een binaire drempel bij 10 m/s 
consistent met de mediaan bezettingsstijging van +5.3% 
gevonden in H-E6. `temp_c` werd expliciet uitgesloten: de 
post-hoc Simpson's Paradox analyse (nb07) toonde een globale 
ρ = −0.013 en een partiële ρ = −0.050 na seizoenscorrectie — 
het schijnbare temperatuureffect is volledig medieerd door de 
seizoensencoding in `month_sin/cos`. `wind_strong` is uiterst 
spaarzaam (0.2% positief) en wordt in nb11 geëvalueerd via 
permutatie-importance; bij verwaarloosbare importance wordt 
de feature verwijderd als post-hoc vereenvoudiging.

**Ruimtelijke features (5 features)**  
`mean_occ_by_parking` (target encoding, cv=5) representeert 
de gemiddelde historische bezettingsgraad per parking en 
comprimeert parking-identiteit naar één informatiedense 
continue feature (Pargent et al., 2022). `behavioral_cluster` 
(k=2, silhouette = 0.615, nb06) representeert de 
data-gedreven gedragsclusterindeling en is complementair: 
drie parkings vertonen een gedragsprofiel dat afwijkt van 
hun administratieve tier. `log_capacity` vangt de absolute 
infrastructuuromvang op — conceptueel onderscheiden van de 
bezettingsgraad — op basis van de empirisch vastgestelde 
niet-lineaire relatie tussen parkingomvang en 
bezettingsdynamiek (Caicedo, 2010). `cluster_data` en 
`mean_occ_by_parking` vertonen VIF-waarden net boven 10 
(11.10 en 10.97). Aangezien het predictiemodel boomgebaseerd 
is (Random Forest / XGBoost), introduceert multicollineariteit 
geen schattingsbias; beide features worden behouden als 
complementaire ruimtelijke representaties (Hastie et al., 2009).

**Event-features en interacties (11 features)**  
De vijf event-type dummies (`is_football_day`, `is_festival_day`, 
`is_kermis_day`, `is_carnival_day`, `is_procession_day`) 
worden aangevuld met `n_concurrent_events` voor het modelleren 
van potentieel super-additieve parkeerdruk bij overlappende 
evenementen. De generieke `is_event_day` werd bewust niet 
opgenomen als zelfstandige feature: de Wilcoxon matched-pairs 
analyse (nb07) toonde tegengestelde tier-effecten die bij 
aggregatie naar nul middelen. De vijf interactietermen 
(event-type × `tier_num`) modelleren deze tegengestelde 
effecten expliciet.

**Cascade-features (2 features)**  
`hours_since_last_event` en `hours_to_next_event` modelleren 
de temporele aanloop- en naijleffecten rondom events als 
continue features, superieur aan binaire dagdummies voor 
het vangen van graduele bezettingsdynamiek (H-E3, nb07). 
Rijen vóór het eerste en ná het laatste event per parking 
(13% van de trainingsset) worden geïmputeerd met de 
schildwachtwaarde 999 — wat het model signaleert dat de 
observatie zich ver buiten de eventzone bevindt.

---

### Datakwaliteitsbeperking

De effectieve trainingsset na verwijdering van lag-NaN 
bedraagt 108.519 rijen (83.6% van de ruwe trainingsset). 
De 16.4% uitval is een directe consequentie van de 
tijdreeksgaten geïntroduceerd door de kwaliteitsfilter 
in nb05 en wordt gerapporteerd als een structurele 
datakwaliteitsbeperking. De holdoutset (98.1% volledig) 
is nagenoeg onaangetast; evaluatiemetrieken in nb11 zijn 
representatief voor de volledige 2025-periode.

---

### Referenties

- Caicedo, F. (2010). Real-time parking space availability 
  prediction. *Transportation Research Part C*, 18(6), 1023–1034.
- Cerqueira, V., Torgo, L., & Mozetič, I. (2023). Evaluating 
  time series forecasting models: An empirical study on 
  performance estimation methods. *Machine Learning*, 112, 
  2297–2340.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The 
  Elements of Statistical Learning* (2nd ed.). Springer.
- Lira, B., et al. (2021). Short-term parking occupancy 
  forecasting using machine learning. *Sensors*, 21(5), 1778.
- Pargent, F., et al. (2022). Regularized target encoding 
  outperforms traditional methods in supervised machine 
  learning with high cardinality features. *Computational 
  Statistics*, 37, 2671–2692.
- Wan, R., et al. (2023). Periodic representation and 
  high-frequency temporal patterns in time series. 
  *Proceedings of AAAI*, 37(9), 10638–10646.
